# Day 13 · Pandas 基础

> **前置**:Day11-12 已掌握 NumPy 数组/向量化/统计
> **关键问题**:NumPy 数组只能存同类型数据,如何像 Excel 表格一样存"姓名(string)+年龄(int)+成绩(float)"?Pandas 的 DataFrame 就是"带标签的二维表"。

**引入:从 NumPy 到 Pandas**

抽问:`np.array([1, "a", 3])` 的 dtype 是什么?(答案:`<U11`,全部变成字符串)NumPy 数组只能装同类型数据,但现实数据是异构的 —— 需要"姓名(string)+年龄(int)+成绩(float)"。Pandas 的 DataFrame 就是 Excel 一样的二维表,每列可以有不同数据类型。

---

**第 1 讲:Series —— 带索引的一维数组**

类比 Excel 的一列:有"值"还有"标签"(索引 index)。列表 list 没有标签,Series 给每个元素挂了一个索引,既能按位置访问,也能按标签访问。


In [ ]:
import pandas as pd

# 从列表创建 Series(自动 0-based 索引)
s1 = pd.Series([10, 20, 30])
print(s1)
print(s1.index.tolist())            # [0, 1, 2]

# 自定义索引(a/b/c)
s2 = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s2)
# 按标签访问
print(s2["b"])                      # 20

# 从字典创建 Series(key → index)
data = {"数学": 90, "语文": 85, "英语": 88}
s3 = pd.Series(data)
print(s3)


**DataFrame —— 二维表格**

类比 Excel 一个 sheet:"列优先"字典 —— dict 的 key = 列名,dict 的 value = 列数据(Pandas Series)。每列可以是不同数据类型,这正是 DataFrame 比 NumPy 数组强的地方。


In [ ]:
import pandas as pd

# 从字典创建 DataFrame
data = {
    "姓名": ["张三", "李四", "王五"],
    "年龄": [22, 28, 25],
    "成绩": [88.5, 92.0, 76.5],
}
df = pd.DataFrame(data)
print(df)

# 列选择:单列 → Series
print(df["姓名"])
print(type(df["姓名"]).__name__)    # Series
# 列选择:多列 → DataFrame
print(df[["姓名", "成绩"]])
print(type(df[["姓名", "成绩"]]).__name__)  # DataFrame

# 常用属性
print("shape:", df.shape)           # (行数, 列数)
print("dtypes:")
print(df.dtypes)
print("columns:", df.columns.tolist())


---

**第 2 讲:数据查看 head / tail / info / describe**

拿到一个 DataFrame 先看轮廓,别直接 `print` 整个大表。`head(n)` 看前 n 行(默认5),`tail(n)` 看后 n 行,`describe()` 一行搞定数值列统计,`info()` 告诉你每列有多少非空值、什么类型。


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],
    "年龄": [22, 28, 25, 30, 26, 24],
    "成绩": [88.5, 92.0, 76.5, 95.0, 81.0, 70.0],
})

# 前 3 行
print(df.head(3))
# 后 2 行
print(df.tail(2))
# 数值列统计摘要(计数/均值/标准差/最大最小)
print(df.describe())
# 每列非空数量 + 数据类型
# info() 输出到 stdout,没有返回值
df.info()


**列选择 vs 行选择(loc / iloc)**

Pandas 两套选址器:
- `loc`:**按标签**(label),切片**包含**右端点
- `iloc`:**按位置**(position,0-based),切片**不含**右端点

口诀:"loc 标签含尾,iloc 位置不含尾"。和 Python list 切片 `lst[0:2]` 只取 0、1 对比着记。


In [ ]:
import pandas as pd

# 构造带自定义行标签的 DataFrame
df = pd.DataFrame(
    {
        "姓名": ["张三", "李四", "王五", "赵六"],
        "年龄": [22, 28, 25, 30],
        "城市": ["北京", "上海", "北京", "深圳"],
    },
    index=["x", "y", "z", "w"],
)

# loc:按标签取一行 → Series
print(df.loc["y"])
# loc:标签切片 —— 包含右端点
# 和 Python list 切片不同,loc 含尾!
print(df.loc["x":"z"])

# iloc:按位置取一行(0-based)
print(df.iloc[0])
# iloc:位置切片 —— 不含右端点
# 和 Python list[0:2] 一样,只取 0、1
print(df.iloc[0:2])

# 组合:行+列一起选
print(df.loc["x":"z", ["姓名", "城市"]])
print(df.iloc[0:2, 0:2])


---

**第 3 讲:过滤与排序**

过滤 = 用布尔条件挑行,排序 = 按某列值重排。过滤本质:`df[布尔 Series]`;多条件用 `&`(且)、`|`(或),每个条件必须包在括号里。


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "年龄": [22, 28, 25, 30, 26],
    "城市": ["北京", "上海", "北京", "深圳", "上海"],
    "成绩": [88.5, 92.0, 76.5, 95.0, 81.0],
})

# 单条件过滤:年龄 > 25
print(df[df["年龄"] > 25])

# 多条件:&(且),每个条件必须加括号!
print(df[(df["年龄"] > 25) & (df["城市"] == "北京")])

# query:SQL 风格字符串过滤
print(df.query("年龄 > 25 and 城市 == '北京'"))

# 排序:按单列升序
print(df.sort_values("年龄"))
# 排序:先按城市升序,城市相同按成绩降序
print(df.sort_values(["城市", "年龄"],
                     ascending=[True, False]))


---

**今日小结**

本节围绕 Pandas 两个核心数据结构 —— **Series**(一维带索引)和 **DataFrame**(二维表格),覆盖从创建、查看、选址到过滤排序的完整流程。

**更多练习**

- 当堂练:course/lesson13/in_class/practice01-06.py
- 课后作业:course/lesson13/homework/task01-03.py
